#### RAGAS 라이브러리를 활용한 Golden Dataset 생성 

```text
- AI 에이전트 평가 시스템을 구축할 때 가장 중요한 것 중 하나는 Golden Dataset(정답 데이터셋) 이다.
- Golden Dataset은 에이전트의 응답을 평가할 때 기준이 되는 ground truth(정답) 역할을 함.
```

```text
# 어떻게 가상데이터를 생성하는가

* langchain loader를 활용해서 문서를 기반으로 생성한다
* 문서를 제공해주면, ragas에서 Q-A 데이터로 골든 데이터셋을 생성


# 전체 워크플로우
PDF 문서들 → pyzerox로 마크다운 변환 → 텍스트 파일로 변환 → LangChain으로 로드 → RAGAS로 테스트셋 생성

문서 로딩: LangChain의 TextLoader를 사용하여 문서를 로드하고,
테스트셋 생성: RAGAS의 TestsetGenerator를 사용하여 질문-답변 쌍 생성

```

In [ ]:
"""

? nest_asyncio.apply()

# Jupyter 노트북 환경에서 이미 실행 중인 이벤트 루프 안에서 asyncio.run()을 사용할 수 있게 해줌
# py-zerox 는 비동기로 요청을 하는데, Jupyter의 기존 이벤트 루프와 asyncio 호환을 위해 필요
"""

'\n# Jupyter 노트북 환경에서 이미 실행 중인 이벤트 루프 안에서 asyncio.run()을 사용할 수 있게 해줌\n# py-zerox 는 비동기로 요청을 하는데, Jupyter의 기존 이벤트 루프와 asyncio 호환을 위해 필요\n'

In [ ]:
import nest_asyncio

from dotenv import load_dotenv

nest_asyncio.apply()  
load_dotenv()

False

```text

- 입력: documents_with_english_titles/ 폴더의 PDF 파일들
- 출력: documents_with_english_titles_markdown/ 폴더에 .md 파일 생성

```

In [ ]:
"""
아래 코드는 토큰 비용절감을 위해서 주석처리를 함 (langgraph-agent 코드와 py-zerox 공식문서 참고)

- 사용할 모델은 gpt-4.1 -> gpt-5 는 추론 모델이기 때문에 복잡한 작업에 성능이 좋지만, 현 시점에서의 작업은 사진 인식만 하면 되기 때문에 굳이 비싼 모델을 쓸 필요는 없음
"""
# from pyzerox import zerox
# import os, sys
# import asyncio


# model = 'gpt-4.1'  # pyzerox가 PDF를 해석할 때 사용할 LLM 모델 (vision 이 가능한 모델, 가성비)

# async def main():
#     """
#     PDF를 마크다운으로 변환
#     """

#     kwargs = {}
#     custom_system_prompt = None  # 기본 시스템 프롬프트 사용
#     select_pages = None  # None이면 모든 페이지를 변환
#     output_dir = 'documents_with_english_titles_markdown'  # 변환 결과 저장 디렉토리

#     # 폴더 내 모든 PDF 파일을 순회하며 변환 (!! 실행 전 경로 확인 필수)
#     for file_path in os.listdir("./documents_with_english_titles"):
#         file_path = os.path.join("./documents_with_english_titles", file_path)
#         # PDF를 마크다운으로 변환 (비동기 함수 - LLM API 호출 포함)
#         result = await zerox(
#             file_path=file_path,  # pdf 파일 위치
#             model=model,
#             output_dir=output_dir,
#             custom_system_prompt=custom_system_prompt,  # 파싱하고자 하는 문서에 대한 부연설명 (생략해도 됨)
#             select_pages=select_pages,  
#             **kwargs
#         )
#         print(f"Converted {file_path} to markdown.")
#         print(f'result == {result}')

# # asyncio.run()으로 비동기 함수 실행 (nest_asyncio로 인해서 Jupyter에서도 비동기 함수가 동작할 수 있게 됨)
# result = asyncio.run(main()) 

In [11]:
from pathlib import Path

print(Path.cwd())  # 상대 경로는 현재 작업 디렉터리를 기준으로 해석

c:\studyhub\ai\agentic-ai\ai-agent-evaluation\notebook


#### Text 파일로 변환

```text
RAGAS의 TestsetGenerator가 요구하는 입력 형식: .txt 파일 (.md -> .txt 확장자만 변경)
```

In [12]:
input_dir = Path('../documents/documents_with_english_titles_markdown')
output_dir = Path('../documents/documents_with_english_titles_txt')  # 텍스트 파일을 저장할 디렉토리

# 출력 디렉토리가 없으면 생성
output_dir.mkdir(exist_ok=True)


# 모든 마크다운 파일을 텍스트 파일로 변환
for md_file in input_dir.glob("*.md"):
    # 마크다운 파일 내용 읽기 (UTF-8 인코딩)
    text_content = md_file.read_text(encoding='utf-8')
    
    # 확장자만 .txt로 변경하여 저장 (내용은 동일)
    txt_file = output_dir / md_file.with_suffix('.txt').name
    txt_file.write_text(text_content, encoding='utf-8')

In [ ]:
"""
* 에러 코드 - 노션 참고

    loader = DirectoryLoader('../documents/documents_with_english_titles_txt', glob='**/*.txt')
    documents = loader.load()
"""

"\n* 에러 코드\n\n    loader = DirectoryLoader('../documents/documents_with_english_titles_txt', glob='**/*.txt')\n    documents = loader.load()\n"

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    '../documents/documents_with_english_titles_txt', 
    glob='**/*.txt',
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'})
documents = loader.load()

In [20]:
documents

[Document(metadata={'source': '..\\documents\\documents_with_english_titles_txt\\delegation_of_authority.txt'}, page_content='전결 규정 (Delegation of Authority)\n====================================\n\n제 1장 전결 규정 개요\n-------------------\n\n제 1조 (목적)\n- 본 규정은 회사 내 의사 결정의 신속성, 책임성을 확보하기 위하여 전결 권한을 명확히 정하고자 한다.\n\n제 2조 (적용 범위)\n- 본 규정은 모든 부서 및 직원에게 적용되며, 별도 규정이 있는 경우 해당 규정을 우선으로 한다.\n\n제 2장 전결 권한 분류\n-------------------\n\n제 3조 (전결 권한 표)\n\n| 전결사항              | 팀장   | 부서장 | 본부장   | 대표이사   |\n|----------------------|--------|--------|----------|------------|\n| 인사(채용, 승진, 징계)  | 검토   | 승인   | 승인     | 최종 승인  |\n| 직원 연봉 조정         | 검토   | 승인   | 승인     | 최종 승인  |\n| 연차 및 휴가 승인      | 승인   | 승인   | 보고     | -          |\n| 예산 승인 (50만원 이하)  | 승인   | 승인   | 보고     | -          |\n| 예산 승인 (50만원 초과~200만원) | 검토 | 승인 | 최종 승인 | -        |\n| 예산 승인 (200만원 초과) | 검토   | 검토   | 승인     | 최종 승인  |\n| 출장비 승인           | 승인   | 승인   | 보고     | -          |\n| 법인카드 사용 승인     | 승인   | 승인   | 보고     | -       

### RAGAS 패키지

##### 1. 모델별 실행 문제 및 대체 방안

| 모델 | 발생 현상 및 원인 | 대응 방향 |
| --- | --- | --- |
| OpenAI | RAGAS가 내부적으로 구성한 에이전트와 자체 개발한 프롬프트로 응답을 생성하는 과정에서, JSON 형식으로 받은 response의  파싱·후처리 오류 발생 | 현재는 정상 동작할 가능성도 있으므로 재현 여부 확인 필요 |
| Anthropic | OpenAI 사용 중 발생한 오류를 우회하기 위한 대체 모델 | Anthropic 모델로 대체하여 개발 진행 |
| Gemini | API 요청 제한(Rate Limit) 발생 | 요청 제한을 고려한 실행 방식 필요 |

##### 2. 영어 전용 출력 및 한국어 데이터 문제

- 한국어 문서를 기반으로 QA(질문·답변) 쌍을 생성했으나, 문서 출처에 해당하는 한국어 내용의 앞뒤에 영어 설명이 붙는 현상 발생.
- 영어 설명의 혼합이 평가 성능에는 영향을 주지 않을 수도 있으나, 생성된 데이터셋의 신뢰도를 떨어뜨릴 수 있음.


```text
RAGAS의 TestsetGenerator에는 두 가지 모델을 요구한다

1. Generator LLM: 질문-답변 쌍을 생성하는 언어 모델 (여기서는 Claude 사용)
2. Embedding 모델: 문서 간 의미적 유사도를 계산하는 임베딩 모델 (여기서는 OpenAI 사용)
3. LiteLLM을 사용하면 다양한 LLM 제공자(OpenAI, Anthropic 등)를 통합된 인터페이스로 호출할 수 있다.

```

##### llm_factory()와 LiteLLM의 역할

- llm_factory()
  - LLM을 RAGAS에서 사용할 수 있는 형태로 래핑하는 함수.
  - 현재 설정에서는 LiteLLM 프로바이더를 사용해 모델을 연동함.

- LiteLLM
  - OpenAI, Amazon Bedrock 등 다양한 모델 제공자를 통합된 인터페이스로 연결하는 라이브러리.
  - 모델 라우팅(Routing)과 대체 모델 전환(Fallback)을 구성할 때 활용함.

| 기능 | 역할 |
| --- | --- |
| Routing | 설정된 조건과 정책에 따라 요청을 적절한 모델이나 제공자로 전달 |
| Fallback | 모델 호출 실패나 서비스 장애 등 예상치 못한 상황에서 대체 모델로 전환 |

여러 모델을 일관된 방식으로 연동하고, 장애 상황에서도 모델 교체가 원활하도록 구성하기 위함이다.
단, Routing과 Fallback은 별도 설정이 필요하며, llm_factory() 사용만으로 자동 적용되지는 않음.


In [ ]:
"""
이 셀을 돌리면 ragas_testset.csv 가 나온다
RAGAS 라이브러리의 에러 이슈로 주석
"""
# # LLM 및 임베딩 모델 설정
# # RAGAS 테스트셋 생성에 필요한 모델들을 초기화한다.

# import litellm  # 다양한 LLM 제공자를 통합하는 라이브러리
# import openai
# from ragas.llms import llm_factory  # RAGAS용 LLM 래퍼 생성
# from ragas.embeddings import OpenAIEmbeddings  # 임베딩 모델 래퍼

# # LLM 설정 (Claude 사용)
# # LiteLLM을 통해 Anthropic의 Claude 모델을 사용
# generator_llm = llm_factory(
#     "anthropic/claude-sonnet-4-5",  # LiteLLM 형식: "제공자/모델명"
#     provider="litellm",  # LiteLLM 프로바이더 사용
#     client=litellm.completion,  # LiteLLM의 completion 함수 전달
#     temperature=0.1,  # 낮은 temperature = 더 일관된 출력
#     top_p=None,  # temperature와 충돌 방지를 위해 비활성화
#     max_tokens=64000,  # 최대 출력 토큰 수
# )

# # 임베딩 모델 설정 (OpenAI 사용(anthropic은 임베딩이 없음))
# # 문서 간 의미적 유사도 계산을 위한 임베딩 모델
# # text-embedding-3-large는 OpenAI의 최신 고성능 임베딩 모델임

# openai_client = openai.OpenAI()  # OpenAI 클라이언트 초기화
# generator_embeddings = OpenAIEmbeddings(
#     client=openai_client,
#     model="text-embedding-3-large"  # 고품질 임베딩
# )

In [30]:
import pandas as pd

df_ragas = pd.read_csv('../documents/ragas_testset.csv', encoding='utf-8')
df_ragas.head(10)

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What are the working hours specified in 제 6조?,['제 3장 근로 시간 및 휴가\n\n### 제 6조 (근로 시간)\n- 기본 근무...,"According to 제 6조 (근로 시간), the basic working h...",Sarah Chen,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
1,"As an HR Manger, can you explane what is the p...",['제 4 장 급여 및 복리 후생\n\n## 제 11 조 (급여 지급)\n- 급여는...,According to the company's benefits policy out...,Sarah Chen,MISSPELLED,LONG,single_hop_specific_query_synthesizer
2,hey so i need know about this 징계위원회 thing in o...,['제 5 장 직장 내 규율\n\n## 제 14 조 (기밀 유지)\n- 회사 기밀 ...,"Based on the company policy, the 징계위원회 (discip...",Sarah Chen,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
3,company health checkup X-ray coverage benefits,['# 직원 복리후생 및 복지 가이드\n\n## 제 1 장 복리후생 개요\n\n#...,"According to the employee benefits guide, the ...",Sarah Chen,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
4,"As an HR Manger, I need to undrestand how anua...",['<1-hop>\n\n# 직원 복리후생 및 복지 FAQ\n\n 1. 연차 유급 휴...,Based on the company's employee benefits and w...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
5,How do the company's flexible work arrangement...,['<1-hop>\n\n# 직원 복리후생 및 복지 FAQ\n\n 1. 연차 유급 휴...,The company's flexible work arrangements and l...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
6,Hi i need know about how many days of annual p...,['<1-hop>\n\n# 직원 복리후생 및 복지 FAQ\n\n 1. 연차 유급 휴...,"Based on the company policies, here is the inf...",NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
7,how do flexible work arrangements and annual l...,['<1-hop>\n\n# 직원 복리후생 및 복지 FAQ\n\n 1. 연차 유급 휴...,The company's flexible work arrangements and a...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
8,회사에서 직장 내 성희롱이나 괴롭힘 발생 시 징계위원회는 어떤 역할을 하며 징계 절...,['<1-hop>\n\n제 3장 법적 의무\n\n## 제 7조 (근로 기준법 준수)...,회사에서 직장 내 성희롱이나 괴롭힘이 발생하면 체계적인 징계 절차가 진행됩니다. 먼...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,연차 휴가 15일 받는 조건,['<1-hop>\n\n# 직원 복리후생 및 복지 FAQ\n\n 1. 연차 유급 휴...,연차 휴가 15일을 받기 위해서는 근속 연수가 1~3년이어야 합니다. 근속 연수에 ...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer


In [ ]:
"""
# RAGAS; 추후 패키지 업데이트가 된다면, 코드 실행해볼 것
- user_input → 앞에 영어로 질문을 생성하고, 
- reference contexts → 문서 출처는 잘 넣어줌
- reference → 원본 문서가 들어가야 하는데 영번역이 들어감
- persona_name → sarah chen 은 문서에 없음
- query_style → misspelled는 잘못 들어간 데이터임 (사내 문서는 오타가 있을 수 없음)
"""

'\n# RAGAS; 추후 패키지 업데이트가 된다면, 코드 실행해볼 것\n- user_input → 앞에 영어로 질문을 생성하고, \n- reference contexts → 문서 출처는 잘 넣어줌\n- reference → 원본 문서가 들어가야 하는데 영번역이 들어감\n- persona_name → sarah chen 은 문서에 없음\n- query_style → misspelled는 잘못 들어간 데이터임 (사내 문서는 오타가 있을 수 없음)\n'